# 🧠 MindBridge AI — Dedicated Remote GPU Inference Server

This notebook turns **Google Colab (Free T4 GPU / 15 GB VRAM)** into a dedicated, high-speed remote model inference server for your **MindBridge AI** project.

### 🎯 How it works:
1. **Colab GPU** loads **Meta Llama-3-8B-Instruct** in 4-bit NF4 quantization.
2. It attaches your fine-tuned LoRA adapter: **`aayush0K/Llama-3-MindBridge`**.
3. It creates a secure, public **HTTPS endpoint** using ngrok or Cloudflare tunnel.
4. Your **Local PC** (Next.js + FastAPI + PostgreSQL + Redis) sends prompt requests to this Colab GPU and receives fine-tuned empathetic responses in real-time.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pydantic pyngrok
!pip install -q torch transformers peft bitsandbytes accelerate huggingface_hub

import torch
print('=' * 60)
print('CUDA Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('VRAM Total     :', f'{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('=' * 60)

## 🔑 Step 2: Authenticate with Hugging Face (Required for Meta Llama 3)
Sign in with your Hugging Face Access Token.

In [ ]:
from huggingface_hub import login
login()
print('✅ Hugging Face authenticated successfully!')

## 🚀 Step 3: Start the Remote GPU Inference Server & Public Tunnel

Paste your free ngrok Authtoken from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) into `NGROK_AUTH_TOKEN` below and run the cell.

In [ ]:
import os
import time
import threading
import torch
import uvicorn
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from pyngrok import ngrok

# ── CONFIGURATION ──────────────────────────────────────────────────────────
BASE_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
ADAPTER_ID = 'aayush0K/Llama-3-MindBridge'
NGROK_AUTH_TOKEN = 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE'  # Free from dashboard.ngrok.com
PORT = 8001

# ── 1. LOAD MODEL IN 4-BIT NF4 ─────────────────────────────────────────────
print('=' * 65)
print('🧠 Loading Base Llama 3 8B + MindBridge LoRA on GPU...')
print('=' * 65)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print('📦 Step 1: Loading Base Model in 4-bit...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print(f'📦 Step 2: Attaching LoRA Adapter from Hugging Face: {ADAPTER_ID}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()
print('✅ MindBridge Model and Adapter successfully loaded into GPU memory!')

# ── 2. FASTAPI SERVER SETUP ────────────────────────────────────────────────
app = FastAPI(title='MindBridge Remote GPU Inference Server')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class ChatMessage(BaseModel):
    role: str
    content: str

class InferenceRequest(BaseModel):
    messages: Optional[List[ChatMessage]] = None
    message: Optional[str] = None
    prompt: Optional[str] = None
    temperature: float = Field(default=0.6, ge=0.0, le=2.0)
    max_new_tokens: int = Field(default=300, ge=1, le=2048)

class InferenceResponse(BaseModel):
    response: str
    model: str
    generation_time_ms: float

@app.get('/')
def root():
    return {
        'service': 'MindBridge AI Remote GPU Inference Service',
        'status': 'ready',
        'base_model': BASE_MODEL,
        'adapter': ADAPTER_ID,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None',
        'attribution': 'Built with Meta Llama 3'
    }

@app.get('/health')
@app.get('/model/status')
def health():
    vram = round(torch.cuda.memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0.0
    return {
        'base_model': BASE_MODEL,
        'adapter': ADAPTER_ID,
        'model_loaded': True,
        'adapter_loaded': True,
        'cuda_available': torch.cuda.is_available(),
        'device': 'cuda',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None',
        'quantization': '4-bit NF4',
        'vram_allocated_gb': vram
    }

@app.post('/generate', response_model=InferenceResponse)
def generate(request: InferenceRequest):
    start_t = time.time()
    if request.messages:
        messages_dict = [{'role': m.role, 'content': m.content} for m in request.messages]
    elif request.message:
        messages_dict = [
            {'role': 'system', 'content': 'You are MindBridge, an empathetic, supportive mental health AI companion. Provide warm, non-judgmental validation and practical coping strategies without medical diagnoses or prescriptions.'},
            {'role': 'user', 'content': request.message}
        ]
    elif request.prompt:
        messages_dict = [
            {'role': 'system', 'content': 'You are MindBridge, an empathetic, supportive mental health AI companion. Provide warm, non-judgmental validation and practical coping strategies without medical diagnoses or prescriptions.'},
            {'role': 'user', 'content': request.prompt}
        ]
    else:
        raise HTTPException(status_code=422, detail="Missing 'messages', 'message', or 'prompt'.")

    prompt_str = tokenizer.apply_chat_template(messages_dict, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            temperature=request.temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    gen_time_ms = (time.time() - start_t) * 1000

    return InferenceResponse(
        response=generated_text.strip(),
        model=f'Llama-3-MindBridge ({ADAPTER_ID})',
        generation_time_ms=gen_time_ms
    )

# ── 3. START UVICORN IN BACKGROUND THREAD (Colab-Safe) ────────────────────
def run_uvicorn():
    config = uvicorn.Config(app=app, host='0.0.0.0', port=PORT, log_level='info')
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# ── 4. CREATE PUBLIC HTTPS TUNNEL ─────────────────────────────────────────
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN.strip() and NGROK_AUTH_TOKEN != 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE':
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
    public_tunnel = ngrok.connect(PORT)
    print('\n' + '=' * 65)
    print('🎉 YOUR REMOTE GPU INFERENCE SERVER IS LIVE!')
    print('=' * 65)
    print(f'🔗 Public Base URL   : {public_tunnel.public_url}')
    print(f'🔗 Inference Endpoint: {public_tunnel.public_url}/generate')
    print('=' * 65)
    print('\n👉 In your local PC backend/.env, update:')
    print(f'INFERENCE_URL={public_tunnel.public_url}/generate\n')
else:
    print('\n⚠️ NGROK_AUTH_TOKEN was not provided.')
    print('Get your free token from https://dashboard.ngrok.com/get-started/your-authtoken')
    print('and paste it into NGROK_AUTH_TOKEN = "..." above to generate a public HTTPS URL.')

# Keep thread alive
try:
    server_thread.join()
except KeyboardInterrupt:
    print('🛑 Server stopped.')